# 🎓 LoRA Fine-tuning v1 (Tutorial-based / KcBERT)

**보정된 UnSmile 데이터만** 사용하여 학습합니다.

- **모델**: `beomi/kcbert-base`
- **메트릭**: `abuse_recall` (통일)
- **데이터**: UnSmile 보정 14,690건

In [1]:
import os, torch, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import precision_recall_fscore_support, label_ranking_average_precision_score
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA: True
GPU: NVIDIA L40S


In [2]:
MODEL_NAME = "beomi/kcbert-base"
OUTPUT_DIR = "./output/lora_tutorial_kcbert"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 통일된 하이퍼파라미터
EPOCHS, BATCH_SIZE, LEARNING_RATE = 10, 32, 2e-4
MAX_LENGTH = 128
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.1

LABEL_NAMES = ["여성/가족", "남성", "성소수자", "인종/국적", "연령", "지역", "종교", "기타 혐오", "악플/욕설", "clean"]
NUM_LABELS = len(LABEL_NAMES)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
train_df = pd.read_csv("../../3_UnSmile_Correction/unsmile_train_corrected.tsv", sep='\t')
valid_df = pd.read_csv("../../3_UnSmile_Correction/unsmile_valid_corrected.tsv", sep='\t')
print(f"✅ Train: {len(train_df)}건, Valid: {len(valid_df)}건")

✅ Train: 14690건, Valid: 3663건


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    tokenized = tokenizer(examples['문장'], padding='max_length', truncation=True, max_length=MAX_LENGTH)
    tokenized['labels'] = [[float(examples[col][i]) for col in LABEL_NAMES] for i in range(len(examples['문장']))]
    return tokenized

train_dataset = Dataset.from_pandas(train_df).map(preprocess_function, batched=True, remove_columns=train_df.columns.tolist())
valid_dataset = Dataset.from_pandas(valid_df).map(preprocess_function, batched=True, remove_columns=valid_df.columns.tolist())

Map:   0%|          | 0/14690 [00:00<?, ? examples/s]

Map:   0%|          | 0/3663 [00:00<?, ? examples/s]

In [5]:
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS, problem_type="multi_label_classification")
peft_config = LoraConfig(task_type=TaskType.SEQ_CLS, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, target_modules=["query", "key", "value"], bias="none")
model = get_peft_model(base_model, peft_config).to(DEVICE)
model.print_trainable_parameters()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 892,426 || all params: 109,818,644 || trainable%: 0.8126


In [6]:
# 통일된 compute_metrics (abuse_recall 포함)
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(predictions)).numpy()
    preds = (probs > 0.5).astype(int)
    labels_int = labels.astype(int)
    _, abuse_r, abuse_f1, _ = precision_recall_fscore_support(labels_int[:,8], preds[:,8], average='binary', zero_division=0)
    _, clean_r, clean_f1, _ = precision_recall_fscore_support(labels_int[:,9], preds[:,9], average='binary', zero_division=0)
    return {'lrap': label_ranking_average_precision_score(labels, predictions), 'abuse_recall': abuse_r, 'abuse_f1': abuse_f1, 'clean_recall': clean_r, 'clean_f1': clean_f1}

In [7]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, warmup_ratio=0.1, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="abuse_recall", greater_is_better=True,
    logging_steps=50, save_total_limit=2, report_to="none", fp16=True
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=valid_dataset,
                  tokenizer=tokenizer, compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])

In [8]:
print("🚀 학습 시작...")
trainer.train()
print("학습 완료!")

🚀 학습 시작...


Epoch,Training Loss,Validation Loss,Lrap,Abuse Recall,Abuse F1,Clean Recall,Clean F1
1,0.329400,0.275358,0.636476,0.000000,0.000000,0.000000,0.000000
2,0.209800,0.166938,0.826862,0.410553,0.538851,0.679570,0.704571
3,0.162100,0.144718,0.848182,0.442728,0.575732,0.706452,0.730000
4,0.139200,0.133161,0.862925,0.601030,0.660071,0.704301,0.735955
5,0.127500,0.129568,0.867761,0.653797,0.684175,0.689247,0.733829
6,0.121000,0.126900,0.869956,0.607465,0.663854,0.761290,0.753994
7,0.114400,0.125759,0.871062,0.660232,0.684000,0.718280,0.743875
8,0.112400,0.124358,0.873629,0.661519,0.693657,0.694624,0.734091
9,0.111300,0.123738,0.874215,0.628057,0.680614,0.753763,0.754169
10,0.108400,0.123672,0.875023,0.644788,0.686772,0.733333,0.751515


학습 완료!


In [9]:
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(f"{OUTPUT_DIR}/merged_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/merged_model")
print("✅ 모델 저장 완료!")

eval_results = trainer.evaluate()
for k, v in eval_results.items(): print(f"  {k}: {v:.4f}")

✅ 모델 저장 완료!


  eval_loss: 0.1244
  eval_lrap: 0.8736
  eval_abuse_recall: 0.6615
  eval_abuse_f1: 0.6937
  eval_clean_recall: 0.6946
  eval_clean_f1: 0.7341
  eval_runtime: 2.6214
  eval_samples_per_second: 1397.3610
  eval_steps_per_second: 11.0630
  epoch: 10.0000
